In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
import os

In [2]:
BASE_DIR = 'skin_dataset'  # your dataset folder
PROBLEM_CLASSES = ['Acne', 'Darkspots', 'Redness', 'Wrinkles']

for cls in PROBLEM_CLASSES:
    path = os.path.join(BASE_DIR, cls)
    if os.path.exists(path):
        num_images = len([f for f in os.listdir(path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        print(f"{cls}: {num_images} images")
    else:
        print(f"{cls} folder not found!")

Acne: 30 images
Darkspots: 30 images
Redness: 30 images
Wrinkles: 30 images


In [3]:
# %% user settings
BASE_DIR = 'skin_dataset' 
PROBLEM_CLASSES = ['Acne', 'Darkspots', 'Redness', 'Wrinkles']
IMAGE_SIZE = (224,224)
BATCH_SIZE = 32
EPOCHS = 30
MODEL_OUT = 'model_problem.h5'


# %% data generators
train_datagen = ImageDataGenerator(
rescale=1./255,
rotation_range=20,
width_shift_range=0.1,
height_shift_range=0.1,
zoom_range=0.2,
horizontal_flip=True,
validation_split=0.2
)


train_gen = train_datagen.flow_from_directory(
BASE_DIR,
classes=PROBLEM_CLASSES,
target_size=IMAGE_SIZE,
batch_size=BATCH_SIZE,
class_mode='categorical',
subset='training'
)


val_gen = train_datagen.flow_from_directory(
BASE_DIR,
classes=PROBLEM_CLASSES,
target_size=IMAGE_SIZE,
batch_size=BATCH_SIZE,
class_mode='categorical',
subset='validation'
)


# %% model
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(*IMAGE_SIZE,3))
base.trainable = False
x = layers.GlobalAveragePooling2D()(base.output)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(len(PROBLEM_CLASSES), activation='softmax')(x)
model = models.Model(inputs=base.input, outputs=output)


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


# %% train
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS)


# %% save
model.save(MODEL_OUT)

# %% save label mapping
import json

label_map = train_gen.class_indices
with open('label_map_problem.json', 'w') as f:
    json.dump(label_map, f)

print('Saved', MODEL_OUT)


Found 96 images belonging to 4 classes.
Found 24 images belonging to 4 classes.


C:\Users\A S U S\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.3086 - loss: 1.5343 - val_accuracy: 0.4167 - val_loss: 1.2964
Epoch 2/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.6615 - loss: 0.9808 - val_accuracy: 0.5000 - val_loss: 1.0086
Epoch 3/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 1s/step - accuracy: 0.7005 - loss: 0.7359 - val_accuracy: 0.6667 - val_loss: 0.8389
Epoch 4/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.7708 - loss: 0.5361 - val_accuracy: 0.5833 - val_loss: 1.1982
Epoch 5/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.8190 - loss: 0.4451 - val_accuracy: 0.6250 - val_loss: 1.1350
Epoch 6/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.8828 - loss: 0.3809 - val_accuracy: 0.7083 - val_loss: 0.8155
Epoch 7/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.9010 - loss: 0.3466 - val_accuracy: 0.6667 - val_loss: 0.7017
Epoch 8/30
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 1s/step - accuracy: 0.9076 - loss: 0.2591 - val_accuracy: 0.6250 - val_loss: 1.0494
Epoch 9/30
3/3 ━

Saved model_problem.h5
